# Sample Table Generation

Generates 50 realistic sample customer records and saves them as a Unity Catalog table (`workspace.default.sample_customers`) for testing.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.sample_customers (
  record_id      BIGINT  COMMENT 'Sequential record identifier (1-50)',
  first_name     STRING  COMMENT 'Customer first name',
  last_name      STRING  COMMENT 'Customer last name',
  email          STRING  COMMENT 'Generated email address',
  city           STRING  COMMENT 'Customer city',
  status         STRING  COMMENT 'One of: active, inactive, pending',
  signup_date    DATE    COMMENT 'Customer signup date',
  lifetime_value DOUBLE  COMMENT 'Synthetic lifetime value in USD'
)
COMMENT '50 sample customer records generated for testing';

In [0]:
from pyspark.sql import functions as F

first_names = ["Aarav", "Bella", "Carlos", "Diana", "Ethan", "Fatima", "Grace", "Hiroshi",
               "Isla", "James", "Kiran", "Lucia", "Marco", "Nadia", "Omar", "Priya",
               "Quinn", "Rohan", "Sofia", "Tomas"]
last_names = ["Anderson", "Bhatt", "Chen", "Dias", "Evans", "Fernandez", "Gupta", "Hansen",
              "Ibrahim", "Johnson", "Khan", "Lopez", "Mehta", "Nguyen", "Okafor", "Patel",
              "Quintero", "Rivera", "Silva", "Turner"]
cities = ["Austin", "Boston", "Chicago", "Denver", "Miami", "Seattle", "Toronto", "Pune"]

# Build 50 records with native Spark expressions (deterministic, no UDFs)
df = (
    spark.range(50)
    .select(
        (F.col("id") + 1).alias("record_id"),
        F.element_at(F.array(*[F.lit(n) for n in first_names]), ((F.col("id") % 20) + 1).cast("int")).alias("first_name"),
        F.element_at(F.array(*[F.lit(n) for n in last_names]), ((F.col("id") * 7 % 20) + 1).cast("int")).alias("last_name"),
        F.element_at(F.array(*[F.lit(c) for c in cities]), ((F.col("id") * 3 % 8) + 1).cast("int")).alias("city"),
        F.element_at(F.array(F.lit("active"), F.lit("inactive"), F.lit("pending")), ((F.col("id") % 3) + 1).cast("int")).alias("status"),
        F.date_add(F.to_date(F.lit("2026-08-01")), (F.col("id") % 90).cast("int")).alias("signup_date"),
        F.round(((F.col("id") * 137) % 940 + 60).cast("double"), 2).alias("lifetime_value"),
    )
    .withColumn(
        "email",
        F.concat(F.lower(F.col("first_name")), F.lit("."), F.lower(F.col("last_name")), F.lit("@example.com")),
    )
)

# Save as a Unity Catalog table (overwrite on re-run)
df.write.mode("overwrite").saveAsTable("workspace.default.sample_customers")

sample_customers = spark.table("workspace.default.sample_customers")
print(f"Row count: {sample_customers.count()}")
display(sample_customers)